# ATLAS Solar Downscaling Tutorial

This notebook downscales solar radiation using a common workflow for all countries.

The notebook automatically selects the processing method:

- If `country == "argentina"`, it uses the split workflow with overlapping latitude tiles.
- For all other countries, it uses the standard full-country workflow.

The notebook expects preprocessed climate inputs from the preprocessing notebooks and DEM inputs from the DEM folder.

## Methodology Description

This notebook applies a Machine Learning based statistical downscaling approach to generate high-resolution climate projections from coarse-resolution CMIP6 climate model outputs. The method relies on a **Multi-Layer Perceptron (MLP)** neural network trained to learn the relationship between large-scale climate model variables and local terrain characteristics.

The downscaling procedure uses a set of predictors derived from both the CMIP6 climate simulations and the Copernicus GLO90 Digital Elevation Model. The target variable depends on the application and may include temperature, precipitation, solar radiation, or wind-related variables. For each target variable, the corresponding CMIP6 predictor is used together with terrain descriptors that represent the influence of local topography on climate conditions.

During the training phase, the MLP is calibrated using the coarse-resolution CMIP6 variable together with the associated topographic predictors. The trained model is subsequently applied to the high-resolution GLO90 grid, allowing the generation of climate projections at a much finer spatial resolution.

Topographic information derived from the Digital Elevation Model is a key component of the methodology. Variables such as elevation, slope, and terrain aspect provide detailed spatial descriptors that are not explicitly resolved by global climate models, enabling the neural network to reproduce local-scale spatial variability driven by terrain characteristics.

This approach preserves the large-scale climate signal provided by the CMIP6 simulations while enhancing its spatial detail, producing high-resolution climate projections suitable for local impact assessments, climate adaptation planning, and sector-specific analyses.

In the current implementation, the predictor set consists of elevation, slope, and transformed aspect variables (sine and cosine of aspect), which allow the neural network to capture terrain-driven variations in local climate conditions while avoiding discontinuities associated with circular angular measurements.

## Step 1. User parameters

Edit only this cell before running the notebook.

Path convention used in this tutorial:

- Preprocessed climate inputs: `../data/processed/{variable}/{country}/`
- DEM inputs for orography and aspect: `../DEMdata/{country}/`
- Downscaling outputs: `../data/downscaling/{target}/{country}/`

`target` is the name used for the downscaled product folder and output variable. For solar radiation, use `ghi`.
`target_name` is the name of the target variable inside the climate model input file. For CMIP-style solar radiation files, this is usually `rsds`.


In [1]:
from pathlib import Path

# ---------------------------------------------------------------------
# Country and variable settings
# ---------------------------------------------------------------------
country = "peru"      # Example: "argentina", "italy", "kenya"
target = "rsds"             # Output product name. For solar radiation, use "rsds".
target_name = "rsds"       # Variable name inside the target climate input file.

# ---------------------------------------------------------------------
# Scenario settings
# ---------------------------------------------------------------------
experiment = "ssp370"      # Example: "historical", "ssp245", "ssp585"
model = "CNRM-ESM2-1"
month = 1                  # Month to downscale, from 1 to 12.

# Period used to train/apply the monthly mean for the selected scenario.
start_period = "2020" #1985, 2020, 2070
end_period   = "2050" #2014, 2050, 2100

# Full available time range in the preprocessed files.
if experiment == "historical":
    start_date = "1985-01"
    end_date = "2014-12"
else:
    start_date = "2015-01"
    end_date = "2100-12"

# ---------------------------------------------------------------------
# Input and output paths
# ---------------------------------------------------------------------
processed_base_path = Path("../data/processed")
dem_input_path = Path(f"../DEMdata/{country}")

target_input_path = processed_base_path / target / country / model / experiment
tas_input_path = processed_base_path / "tas" / country / model / experiment
clt_input_path = processed_base_path / "clt" / country / model / experiment

output_path = Path(f"../data/downscaled_data/{target}/{country}/{model}/{experiment}")
tile_output_path = output_path / "intermediate_tiles"
cache_output_path = output_path / "supporting_intermediate_data"

output_path.mkdir(parents=True, exist_ok=True)
tile_output_path.mkdir(parents=True, exist_ok=True)
cache_output_path.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Input file names
# ---------------------------------------------------------------------
target_file = target_input_path / f"{target_name}_{start_date}_{end_date}_processed.nc"
tas_file = tas_input_path / f"tas_{start_date}_{end_date}_processed.nc"
clt_file = clt_input_path / f"clt_{start_date}_{end_date}_processed.nc"

era5land_orography_file = dem_input_path / f"era5land_orography_{country}.nc"
era5land_aspect_file = dem_input_path / f"era5land_aspect_{country}.nc"
glo90_orography_file = dem_input_path / f"glo90_orography_{country}.nc"
glo90_aspect_file = dem_input_path / f"glo90_aspect_{country}.nc"

output_variable_name = f"{target}_downscaled"
final_output_file = output_path / f"{output_variable_name}_{country}_m{month}_{start_period}_{end_period}.nc"


## Step 2. Import libraries

In [2]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

import os
import warnings

import numpy as np
import pandas as pd
import rioxarray
import xarray as xr

warnings.filterwarnings("ignore")


## Step 3. Utility functions

These functions are used by both workflows.


In [3]:
def round_coords(xdf, digits=None):
    """Round latitude and longitude coordinates to avoid small grid mismatches."""
    if digits is None:
        digits = 1
    xdf["latitude"] = xdf.latitude.round(digits).astype("float32")
    xdf["longitude"] = xdf.longitude.round(digits).astype("float32")
    return xdf


def era5scaler(xdf, features, target_column):
    """Scale ERA5-Land features and target values."""
    featurescaler = StandardScaler()
    X_scaled = featurescaler.fit_transform(xdf[features])

    targetscaler = StandardScaler()
    y_scaled = targetscaler.fit_transform(xdf[[target_column]]).ravel()

    return targetscaler, X_scaled, y_scaled


def glo90scaler(xdf, features):
    """Scale GLO-90 features before prediction."""
    scaler_glo90 = StandardScaler()
    return scaler_glo90.fit_transform(xdf[features])


def drop_vars_if_present(ds, varnames):
    """Drop variables only if they exist in the Dataset/DataArray."""
    existing = [v for v in varnames if v in ds.variables]
    return ds.drop_vars(existing) if existing else ds


def save_xarray_netcdf_fast(ds, output_file, chunksizes=(1024, 1024), show_progress=True):
    """Save an xarray object to NetCDF using compression."""
    from dask.diagnostics import ProgressBar

    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    tmp_file = output_file.with_name(output_file.stem + "_tmp.nc")
    if tmp_file.exists():
        tmp_file.unlink()

    encoding = {}
    for var in ds.data_vars:
        var_encoding = {
            "dtype": "float32",
            "zlib": True,
            "complevel": 1,
            "shuffle": True,
        }

        if chunksizes is not None:
            shape = ds[var].shape
            safe_chunks = tuple(min(dim_size, chunk_size) for dim_size, chunk_size in zip(shape, chunksizes))
            var_encoding["chunksizes"] = safe_chunks

        encoding[var] = var_encoding

    delayed = ds.to_netcdf(
        tmp_file,
        engine="h5netcdf",
        encoding=encoding,
        compute=False,
        mode="w",
    )

    if show_progress:
        with ProgressBar():
            delayed.compute(scheduler="single-threaded")
    else:
        delayed.compute(scheduler="single-threaded")

    os.replace(tmp_file, output_file)
    return output_file


## Step 4. Functions for model training

The model is trained on the lower-resolution inputs, then applied to the high-resolution GLO-90 predictors.


In [4]:
def make_dataset_era5land(target_var, orography, aspect, tas, clt):
    """Merge target, orography, aspect, temperature and cloud cover into a training dataframe."""
    target_var_interp = target_var.interp(
        latitude=orography.latitude.values,
        longitude=orography.longitude.values,
        method="slinear",
    )
    aspect_interp = aspect.interp(
        latitude=orography.latitude.values,
        longitude=orography.longitude.values,
        method="nearest",
    )
    tas_interp = tas.interp(
        latitude=orography.latitude.values,
        longitude=orography.longitude.values,
        method="slinear",
    )
    clt_interp = clt.interp(
        latitude=orography.latitude.values,
        longitude=orography.longitude.values,
        method="slinear",
    )

    tas_monthly = tas_interp.groupby(tas_interp.time.dt.month).mean()
    clt_monthly = clt_interp.groupby(clt_interp.time.dt.month).mean()
    target_var_monthly = target_var_interp.groupby(target_var_interp.time.dt.month).mean()

    xdf_merged = xr.merge([
        orography,
        aspect_interp,
        tas_monthly,
        clt_monthly,
        target_var_monthly,
    ])

    return xdf_merged.to_dataframe().dropna().reset_index()


def training_ds(era5l_merged, selected_month, target_column):
    """Train the downscaling model for one selected month."""
    columns_to_drop = ["month"]
    if "spatial_ref" in era5l_merged.columns:
        columns_to_drop.append("spatial_ref")

    training_df = era5l_merged.loc[era5l_merged.month == selected_month, :].drop(columns_to_drop, axis=1)
    features = training_df.columns.drop([target_column])

    targetscaler_era5, era5l_X, era5l_y = era5scaler(training_df, features, target_column)
    regr = MLPRegressor(random_state=1, max_iter=1000).fit(era5l_X, era5l_y)

    return features, targetscaler_era5, era5l_X, era5l_y, regr


## Step 5. Functions for GLO-90 feature preparation and prediction

In [5]:
def merge_dataframe(df1, df2, variable):
    """Merge two dataframes using latitude and longitude as keys."""
    return df1.merge(
        df2[["latitude", "longitude", variable]],
        on=["latitude", "longitude"],
        how="inner",
        validate="one_to_one",
    )


def build_glo90_dataframe(orography_glo90, aspect_glo90, tas_monthly_mean, clt_monthly_mean):
    """Create the high-resolution feature dataframe for the final prediction grid."""
    tas_glo90_interp = tas_monthly_mean.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="slinear",
    )
    clt_glo90_interp = clt_monthly_mean.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="slinear",
    )
    aspect_glo90_interp = aspect_glo90.interp(
        latitude=orography_glo90.latitude.values,
        longitude=orography_glo90.longitude.values,
        method="nearest",
    )

    orography_glo90_df = (
        drop_vars_if_present(orography_glo90, {"spatial_ref"})
        .to_dataframe()
        .dropna()
        .reset_index()
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )

    aspect_glo90_df = (
        drop_vars_if_present(aspect_glo90_interp, {"spatial_ref"})
        .to_dataframe()
        .dropna()
        .reset_index()
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )

    df_merged_step1 = merge_dataframe(orography_glo90_df, aspect_glo90_df, "aspect")

    tas_glo90_df = (
        tas_glo90_interp.to_dataframe()
        .dropna()
        .reset_index()
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )
    df_merged_step2 = merge_dataframe(df_merged_step1, tas_glo90_df, "tas")

    clt_glo90_df = (
        clt_glo90_interp.to_dataframe()
        .dropna()
        .reset_index()
        .sort_values(["latitude", "longitude"], ascending=[False, True])
    )

    dataset_glo90 = merge_dataframe(df_merged_step2, clt_glo90_df, "clt")

    duplicated = dataset_glo90.duplicated(subset=["latitude", "longitude"]).sum()
    if duplicated:
        raise ValueError(f"Found {duplicated} duplicated latitude/longitude rows before downscaling.")

    return dataset_glo90


def apply_downscaling(dataset_glo90, features, regr, targetscaler_era5):
    """Apply the trained model to the high-resolution feature dataframe."""
    dataset_glo90 = dataset_glo90.copy()

    missing_features = [feature for feature in features if feature not in dataset_glo90.columns]
    if missing_features:
        raise ValueError(f"Missing GLO-90 features: {missing_features}")

    glo90_X = glo90scaler(dataset_glo90.loc[:, features], features)
    downscaled_std = regr.predict(glo90_X)
    downscaled = targetscaler_era5.inverse_transform(downscaled_std.reshape(-1, 1))

    dataset_glo90[output_variable_name] = downscaled
    output_ds = dataset_glo90.set_index(["latitude", "longitude"])[[output_variable_name]].to_xarray()

    # Convert from W/m² mean flux to kWh/m²/day.
    output_ds[output_variable_name] = output_ds[output_variable_name] * 24 / 1000

    return output_ds.sortby(["latitude", "longitude"])


## Step 6. Split functions for Argentina

Argentina is processed in two overlapping latitude tiles. The final output is merged into one file.


In [6]:
split_argentina_by_lat = True
argentina_split_parts = ("north", "south")
argentina_split_overlap_deg = 1.0


def should_split_by_lat(country_name):
    return split_argentina_by_lat and country_name.lower() == "argentina"


def get_lat_split_from_coords(*datasets):
    """Return a split threshold based on latitude coordinates only."""
    lat_min = min(float(ds["latitude"].min()) for ds in datasets if ds is not None)
    lat_max = max(float(ds["latitude"].max()) for ds in datasets if ds is not None)
    return (lat_min + lat_max) / 2


def _sel_lat_range(ds, lat_min, lat_max):
    """Select a latitude range while preserving the latitude coordinate order."""
    lat = ds["latitude"]
    if bool(lat[0] < lat[-1]):
        return ds.sel(latitude=slice(lat_min, lat_max))
    return ds.sel(latitude=slice(lat_max, lat_min))


def cut_by_lat(ds, lat_split, part, overlap_deg=0.0):
    """Cut a Dataset/DataArray by latitude with optional overlap."""
    full_lat_min = float(ds["latitude"].min())
    full_lat_max = float(ds["latitude"].max())

    if part == "north":
        return _sel_lat_range(ds, max(full_lat_min, lat_split - overlap_deg), full_lat_max)
    if part == "south":
        return _sel_lat_range(ds, full_lat_min, min(full_lat_max, lat_split + overlap_deg))

    raise ValueError("part must be 'north' or 'south'")


def cut_core_by_lat(ds, lat_split, part):
    """Remove the overlap from a tile if a hard merge is required."""
    full_lat_min = float(ds["latitude"].min())
    full_lat_max = float(ds["latitude"].max())

    if part == "north":
        return _sel_lat_range(ds, lat_split, full_lat_max)
    if part == "south":
        return _sel_lat_range(ds, full_lat_min, lat_split)

    raise ValueError("part must be 'north' or 'south'")


def merge_latitude_tiles(ds_south, ds_north, lat_split, overlap_deg=1.0, blend=True):
    """Merge south and north tiles into one country-level output."""
    ds_south = ds_south.sortby("latitude").sortby("longitude")
    ds_north = ds_north.sortby("latitude").sortby("longitude")

    if not blend:
        south_core = cut_core_by_lat(ds_south, lat_split, "south")
        north_core = cut_core_by_lat(ds_north, lat_split, "north")
        ds_full = xr.concat([south_core, north_core], dim="latitude", join="outer")
        return ds_full.sortby("latitude").sortby("longitude")

    lat_union = np.union1d(ds_south.latitude.values, ds_north.latitude.values)
    lon_union = np.union1d(ds_south.longitude.values, ds_north.longitude.values)

    south_u = ds_south.reindex(latitude=lat_union, longitude=lon_union)
    north_u = ds_north.reindex(latitude=lat_union, longitude=lon_union)

    lat0 = lat_split - overlap_deg
    lat1 = lat_split + overlap_deg
    w_north = ((south_u.latitude - lat0) / (lat1 - lat0)).clip(0, 1)

    blended = south_u * (1 - w_north) + north_u * w_north
    ds_full = blended.combine_first(south_u).combine_first(north_u)
    ds_full = ds_full.dropna("latitude", how="all").dropna("longitude", how="all")

    return ds_full.sortby("latitude").sortby("longitude")


## Step 7. Load inputs

This cell opens the preprocessed climate files and DEM files using the anonymous, relative paths defined in Step 1.


In [7]:
orography_era5land_all = xr.open_dataset(era5land_orography_file).sel(latitude=slice(None, None, -1))
aspect_era5land_all = xr.open_dataset(era5land_aspect_file).sel(latitude=slice(None, None, -1))

target_all = xr.open_mfdataset(str(target_file)).sel(latitude=slice(None, None, -1))
target_all = target_all.sel(time=slice(start_period, end_period))

tas_all = xr.open_mfdataset(str(tas_file)).sel(latitude=slice(None, None, -1))
tas_all = tas_all.sel(time=slice(start_period, end_period))

clt_all = xr.open_mfdataset(str(clt_file)).sel(latitude=slice(None, None, -1))
clt_all = clt_all.sel(time=slice(start_period, end_period))

lat_split = get_lat_split_from_coords(target_all, tas_all, clt_all) if should_split_by_lat(country) else None


ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


## Step 8. Main processing function

This function is used for both methods:

- full-country processing for standard countries;
- tile processing for Argentina.


In [8]:
def process_downscaling_tile(tile_name=None):
    """Train and apply downscaling on a full country or on one Argentina latitude tile."""
    if tile_name is None:
        suffix = ""
        orography_era5land = orography_era5land_all
        aspect_era5land = aspect_era5land_all
        target_var = target_all
        tas = tas_all
        clt = clt_all
    else:
        suffix = f"_{tile_name}"
        orography_era5land = cut_by_lat(orography_era5land_all, lat_split, tile_name, argentina_split_overlap_deg)
        aspect_era5land = cut_by_lat(aspect_era5land_all, lat_split, tile_name, argentina_split_overlap_deg)
        target_var = cut_by_lat(target_all, lat_split, tile_name, argentina_split_overlap_deg)
        tas = cut_by_lat(tas_all, lat_split, tile_name, argentina_split_overlap_deg)
        clt = cut_by_lat(clt_all, lat_split, tile_name, argentina_split_overlap_deg)

    era5l_merged = make_dataset_era5land(
        target_var,
        orography_era5land,
        aspect_era5land,
        tas,
        clt,
    )

    features, targetscaler_era5, era5l_X, era5l_y, regr = training_ds(
        era5l_merged,
        month,
        target_name,
    )

    orography_glo90 = xr.open_dataset(glo90_orography_file)
    aspect_glo90 = xr.open_dataset(glo90_aspect_file)

    if tile_name is not None:
        orography_glo90 = cut_by_lat(orography_glo90, lat_split, tile_name, argentina_split_overlap_deg)
        aspect_glo90 = cut_by_lat(aspect_glo90, lat_split, tile_name, argentina_split_overlap_deg)

    tas_monthly_mean = tas.groupby(tas.time.dt.month).mean().sel(month=month)
    clt_monthly_mean = clt.groupby(clt.time.dt.month).mean().sel(month=month)

    tas_monthly_mean = drop_vars_if_present(tas_monthly_mean, {"month"})
    clt_monthly_mean = drop_vars_if_present(clt_monthly_mean, {"month"})

    tas_cache = cache_output_path / f"tas_monthly_mean_{country}{suffix}_month{month}.nc"
    clt_cache = cache_output_path / f"clt_monthly_mean_{country}{suffix}_month{month}.nc"

    save_xarray_netcdf_fast(tas_monthly_mean, tas_cache, show_progress=False)
    save_xarray_netcdf_fast(clt_monthly_mean, clt_cache, show_progress=False)

    tas_monthly_mean = xr.open_dataset(tas_cache)
    clt_monthly_mean = xr.open_dataset(clt_cache)

    dataset_glo90 = build_glo90_dataframe(
        orography_glo90,
        aspect_glo90,
        tas_monthly_mean,
        clt_monthly_mean,
    )

    output_ds = apply_downscaling(dataset_glo90, features, regr, targetscaler_era5)
    output_ds.attrs["country"] = country
    output_ds.attrs["target"] = target
    output_ds.attrs["source_target_variable"] = target_name
    output_ds.attrs["month"] = month
    output_ds.attrs["start_period"] = start_period
    output_ds.attrs["end_period"] = end_period
    output_ds.attrs["method"] = "split_tile" if tile_name is not None else "standard_full_country"

    if lat_split is not None:
        output_ds.attrs["lat_split"] = float(lat_split)
        output_ds.attrs["split_overlap_deg"] = float(argentina_split_overlap_deg)

    if tile_name is not None:
        tile_file = tile_output_path / f"{output_variable_name}_{country}_{tile_name}_m{month}_{start_period}_{end_period}.nc"
        save_xarray_netcdf_fast(output_ds, tile_file)

    return output_ds


## Step 9. Run downscaling

Run this cell to create the final NetCDF file.

For Argentina, the notebook automatically creates north/south intermediate tiles, merges them, and saves one final country-level file.
For all other countries, it saves the full-country output directly.


In [9]:
if should_split_by_lat(country):
    output_ds_north = process_downscaling_tile("north")
    output_ds_south = process_downscaling_tile("south")

    output_ds = merge_latitude_tiles(
        output_ds_south,
        output_ds_north,
        lat_split=lat_split,
        overlap_deg=argentina_split_overlap_deg,
        blend=True,
    )
    output_ds.attrs["method"] = "argentina_split_with_overlap_blending"
else:
    output_ds = process_downscaling_tile()

save_xarray_netcdf_fast(output_ds, final_output_file)

print(f"Saved final output: {final_output_file}")


[########################################] | 100% Completed | 105.29 ms
Saved final output: ../data/downscaled_data/rsds/peru/CNRM-ESM2-1/ssp370/rsds_downscaled_peru_m1_2020_2050.nc


## Step 10. Optional quick check

This cell opens the saved file and shows the dataset structure.


In [10]:
xr.open_dataset(final_output_file)

<xarray.Dataset> Size: 1GB
Dimensions:          (latitude: 21963, longitude: 15180)
Coordinates:
  * latitude         (latitude) float32 88kB -18.34 -18.34 ... -0.04333 -0.0425
  * longitude        (longitude) float32 61kB -81.34 -81.33 ... -68.69 -68.69
Data variables:
    rsds_downscaled  (latitude, longitude) float32 1GB ...
Attributes:
    country:                 peru
    target:                  rsds
    source_target_variable:  rsds
    month:                   1
    start_period:            2020
    end_period:              2050
    method:                  standard_full_country